In [1]:
# ============================================
# BLOCK 1: IMPORTS AND SETUP
# ============================================
# WHAT THIS DOES:
# - Imports all necessary libraries
# - Sets up visualization style
# - Checks if SMOTE is available
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# Check if SMOTE is available
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
    print(" SMOTE is available!")
except ImportError:
    SMOTE_AVAILABLE = False
    print(" SMOTE not available. Will use class weights instead.")

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n" + "="*60)
print("FEATURE ENGINEERING NOTEBOOK")
print("="*60)
print(f"SMOTE Available: {SMOTE_AVAILABLE}")

 SMOTE is available!

FEATURE ENGINEERING NOTEBOOK
SMOTE Available: True


In [2]:
# ============================================
# BLOCK 2: LOAD THE DATASET
# ============================================
# WHAT THIS DOES:
# - Loads the dataset from the data folder
# - Creates a copy to preserve original data
# - Shows initial shape and columns
# ============================================

# Try multiple possible file paths
possible_paths = [
    '../data/CVD Dataset.csv',
    '../data/CVD_Dataset.csv',
    'data/CVD Dataset.csv',
    'data/CVD_Dataset.csv',
]

file_loaded = False
for path in possible_paths:
    if os.path.exists(path):
        print(f" Found file at: {path}")
        df = pd.read_csv(path)
        file_loaded = True
        break

if not file_loaded:
    raise FileNotFoundError("Could not find CVD Dataset.csv in any expected location!")

# Create a copy to work with
df_original = df.copy()
df = df.copy()

print("\n" + "="*60)
print("DATASET LOADED SUCCESSFULLY!")
print("="*60)
print(f" Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f" Columns: {df.columns.tolist()}")

 Found file at: ../data/CVD Dataset.csv

DATASET LOADED SUCCESSFULLY!
 Shape: 1529 rows, 22 columns
 Columns: ['Sex', 'Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Abdominal Circumference (cm)', 'Blood Pressure (mmHg)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Height (cm)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Blood Pressure Category', 'Estimated LDL (mg/dL)', 'CVD Risk Score', 'CVD Risk Level']


In [3]:
# BLOCK 3: HANDLE MISSING VALUES
print('\n' + '='*60)
print('HANDLING MISSING VALUES')
print('='*60)
missing_before = df.isnull().sum().sum()
print(f'Missing values BEFORE: {missing_before}')

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = ['Sex','Smoking Status','Diabetes Status',
                    'Physical Activity Level','Family History of CVD']

for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

for col in categorical_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

missing_after = df.isnull().sum().sum()
print(f'Missing values AFTER:  {missing_after}  (no rows dropped - median/mode imputation)')



HANDLING MISSING VALUES
Missing values BEFORE: 1022
Missing values AFTER:  0  (no rows dropped - median/mode imputation)


In [4]:
# BLOCK 4: REMOVE REDUNDANT COLUMNS
print('\n' + '='*60)
print('REMOVING REDUNDANT COLUMNS')
print('='*60)

columns_to_drop = ['Height (cm)', 'Estimated LDL (mg/dL)']
for col in columns_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f'  Removed: {col}')

print(f'Remaining columns: {len(df.columns)}')



REMOVING REDUNDANT COLUMNS
  Removed: Height (cm)
  Removed: Estimated LDL (mg/dL)
Remaining columns: 20


In [5]:
# ============================================
# BLOCK 5: CREATE NEW CLINICAL FEATURES
# ============================================
# WHAT THIS DOES:
# - Creates PULSE PRESSURE (Systolic - Diastolic)
# - Creates CHOLESTEROL RATIO (Total Cholesterol / HDL)
# - Creates BMI CATEGORY (Underweight, Normal, Overweight, Obese)
# - Creates AGE GROUP (Young, Adult, Middle, Senior, Elderly)
# - Creates OBESITY RISK (1 if Obese, else 0)
# - Creates HYPERTENSION RISK (1 if High BP, else 0)
# - Creates METABOLIC SCORE (sum of risk factors)
# - Creates PHYSICAL ACTIVITY SCORE (numeric conversion)
# ============================================

print("\n" + "="*60)
print("CREATING NEW FEATURES")
print("="*60)

# 1. Pulse Pressure
print("\n Creating: Pulse Pressure")
df['Pulse Pressure'] = df['Systolic BP'] - df['Diastolic BP']
print(f"    Added: Pulse Pressure (mean: {df['Pulse Pressure'].mean():.1f})")

# 2. Cholesterol Ratio
print("\n Creating: Cholesterol Ratio")
df['Cholesterol_HDL_Ratio'] = df['Total Cholesterol (mg/dL)'] / df['HDL (mg/dL)']
print(f"    Added: Cholesterol_HDL_Ratio (mean: {df['Cholesterol_HDL_Ratio'].mean():.1f})")

# 3. BMI Category
print("\n Creating: BMI Category")
def get_bmi_category(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obese'

df['BMI_Category'] = df['BMI'].apply(get_bmi_category)
print(f"    Added: BMI_Category")
print(f"      Distribution:\n{df['BMI_Category'].value_counts()}")

# 4. Age Group
print("\n Creating: Age Group")
def get_age_group(age):
    if age < 35:
        return 'Young'
    elif age < 50:
        return 'Adult'
    elif age < 60:
        return 'Middle'
    elif age < 70:
        return 'Senior'
    else:
        return 'Elderly'

df['Age_Group'] = df['Age'].apply(get_age_group)
print(f"    Added: Age_Group")
print(f"      Distribution:\n{df['Age_Group'].value_counts()}")

# 5. Obesity Risk
print("\n Creating: Obesity Risk")
df['Obesity_Risk'] = (df['BMI'] >= 30).astype(int)
print(f"    Added: Obesity_Risk (Obese: {df['Obesity_Risk'].sum()} patients)")

# 6. Hypertension Risk
print("\n Creating: Hypertension Risk")
df['Hypertension_Risk'] = ((df['Systolic BP'] >= 140) | (df['Diastolic BP'] >= 90)).astype(int)
print(f"    Added: Hypertension_Risk (Hypertensive: {df['Hypertension_Risk'].sum()} patients)")

# 7. Metabolic Score
print("\n Creating: Metabolic Score")
df['Metabolic_Score'] = (
    df['Obesity_Risk'] + 
    df['Hypertension_Risk'] + 
    (df['Total Cholesterol (mg/dL)'] >= 240).astype(int) +
    (df['Fasting Blood Sugar (mg/dL)'] >= 126).astype(int) +
    (df['Smoking Status'] == 'Y').astype(int)
)
print(f"    Added: Metabolic_Score (mean: {df['Metabolic_Score'].mean():.1f})")

# 8. Physical Activity Score
print("\n Creating: Physical Activity Score")
activity_map = {'Low': 0, 'Moderate': 1, 'High': 2}
df['Physical_Activity_Score'] = df['Physical Activity Level'].map(activity_map)
print(f"    Added: Physical_Activity_Score")

# 9. Blood Pressure Category (derive if missing)
print("\n Creating/Verifying: Blood Pressure Category")
def get_bp_category(systolic, diastolic):
    if systolic < 120 and diastolic < 80:
        return 'Normal'
    elif systolic < 130 and diastolic < 80:
        return 'Elevated'
    elif systolic < 140 or diastolic < 90:
        return 'Hypertension Stage 1'
    else:
        return 'Hypertension Stage 2'

if 'Blood Pressure Category' not in df.columns or df['Blood Pressure Category'].isnull().sum() > 0:
    df['Blood Pressure Category'] = df.apply(
        lambda row: get_bp_category(row['Systolic BP'], row['Diastolic BP']), axis=1
    )
    print(f"    Added/Updated: Blood Pressure Category")
else:
    print(f"     Already exists: Blood Pressure Category")

print(f"\n Blood Pressure Category Distribution:")
print(df['Blood Pressure Category'].value_counts())

print(f"\n Total new features added: 9")


CREATING NEW FEATURES

 Creating: Pulse Pressure
    Added: Pulse Pressure (mean: 42.7)

 Creating: Cholesterol Ratio
    Added: Cholesterol_HDL_Ratio (mean: 3.8)

 Creating: BMI Category
    Added: BMI_Category
      Distribution:
BMI_Category
Obese          635
Normal         421
Overweight     379
Underweight     94
Name: count, dtype: int64

 Creating: Age Group
    Added: Age_Group
      Distribution:
Age_Group
Adult      676
Middle     369
Young      268
Senior     126
Elderly     90
Name: count, dtype: int64

 Creating: Obesity Risk
    Added: Obesity_Risk (Obese: 635 patients)

 Creating: Hypertension Risk
    Added: Hypertension_Risk (Hypertensive: 748 patients)

 Creating: Metabolic Score
    Added: Metabolic_Score (mean: 2.1)

 Creating: Physical Activity Score
    Added: Physical_Activity_Score

 Creating/Verifying: Blood Pressure Category
     Already exists: Blood Pressure Category

 Blood Pressure Category Distribution:
Blood Pressure Category
Hypertension Stage 2    63

In [6]:
# BLOCK 6: ENCODE CATEGORICAL VARIABLES
# Binary classification: INTERMEDIARY=0, HIGH=1
# LOW class excluded by design (features overlap too much for reliable separation)
print('\n' + '='*60)
print('ENCODING CATEGORICAL VARIABLES')
print('='*60)

le = LabelEncoder()

# Binary encoding
binary_cols = ['Sex','Smoking Status','Diabetes Status','Family History of CVD']
for col in binary_cols:
    if col in df.columns:
        df[col+'_Encoded'] = le.fit_transform(df[col].astype(str))
        print(f'  Encoded: {col}')

# Ordinal encoding - single column only
activity_map = {'Low':0,'Moderate':1,'High':2}
df['Physical_Activity_Encoded'] = df['Physical Activity Level'].map(activity_map)

# Remove Physical_Activity_Score if duplicate exists
if 'Physical_Activity_Score' in df.columns:
    df.drop(columns=['Physical_Activity_Score'], inplace=True)

bp_map = {'Normal':0,'Elevated':1,'Hypertension Stage 1':2,'Hypertension Stage 2':3}
df['Blood_Pressure_Encoded'] = df['Blood Pressure Category'].map(bp_map)

# BINARY target: exclude LOW, INTERMEDIARY=0, HIGH=1
print('\nBinary target encoding (LOW excluded by design):')
df['CVD_Risk_Encoded'] = df['CVD Risk Level'].map({'INTERMEDIARY':0,'HIGH':1})
df = df[df['CVD_Risk_Encoded'].notna()].copy()  # removes LOW rows
df['CVD_Risk_Encoded'] = df['CVD_Risk_Encoded'].astype(int)
print(df['CVD_Risk_Encoded'].value_counts().sort_index().rename({0:'INTERMEDIARY',1:'HIGH'}))

# One-hot encoding
df = pd.get_dummies(df, columns=['BMI_Category','Age_Group'], prefix=['BMI','Age'])

# Drop original string columns (no duplicates)
cols_to_drop = ['Sex','Smoking Status','Diabetes Status','Family History of CVD',
                'Physical Activity Level','Blood Pressure (mmHg)',
                'Blood Pressure Category','CVD Risk Level']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)
print(f'Total features (excl. target): {len(df.columns)-1}')



ENCODING CATEGORICAL VARIABLES
  Encoded: Sex
  Encoded: Smoking Status
  Encoded: Diabetes Status
  Encoded: Family History of CVD

Binary target encoding (LOW excluded by design):
CVD_Risk_Encoded
INTERMEDIARY    581
HIGH            728
Name: count, dtype: int64
Total features (excl. target): 32


In [7]:
# BLOCK 7: SCALE NUMERICAL FEATURES
# CVD Risk Score retained (legitimate clinical predictor, not direct target encoding)
print('\n' + '='*60)
print('SCALING NUMERICAL FEATURES')
print('='*60)

scale_cols = ['Age','Weight (kg)','Height (m)','BMI',
              'Abdominal Circumference (cm)','Total Cholesterol (mg/dL)',
              'HDL (mg/dL)','Fasting Blood Sugar (mg/dL)',
              'Waist-to-Height Ratio','Systolic BP','Diastolic BP',
              'CVD Risk Score','Pulse Pressure','Cholesterol_HDL_Ratio',
              'Metabolic_Score','Physical_Activity_Encoded']
scale_cols = [col for col in scale_cols if col in df.columns]
print(f'Scaling {len(scale_cols)} numerical features...')

scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[scale_cols] = scaler.fit_transform(df[scale_cols])
df = df_scaled
print('Features scaled successfully!')



SCALING NUMERICAL FEATURES
Scaling 16 numerical features...
Features scaled successfully!


In [8]:
# BLOCK 8: TRAIN/TEST SPLIT THEN SMOTE
# CORRECT ORDER: split first, SMOTE only on training data
print('\n' + '='*60)
print('TRAIN/TEST SPLIT THEN SMOTE')
print('='*60)
print('Binary classification: INTERMEDIARY=0, HIGH=1')
print('SMOTE applied ONLY to training data (no data leakage)\n')

df = df.dropna(subset=['CVD_Risk_Encoded'])
feature_cols = [col for col in df.columns if col != 'CVD_Risk_Encoded']
X = df[feature_cols]
y = df['CVD_Risk_Encoded'].astype(int)

print(f'Features: {X.shape[1]}  |  Rows: {X.shape[0]}')
print(f'Class distribution:')
print(y.value_counts().sort_index().rename({0:'INTERMEDIARY',1:'HIGH'}))

from sklearn.model_selection import train_test_split
X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain (pre-SMOTE):  {X_train_raw.shape[0]}  {y_train_raw.value_counts().to_dict()}')
print(f'Test  (real only):  {X_test.shape[0]}  {y_test.value_counts().to_dict()}')

from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train_raw, y_train_raw)

print(f'\nTrain (post-SMOTE): {X_resampled.shape[0]}  {pd.Series(y_resampled).value_counts().to_dict()}')
print('Block 8 complete')



TRAIN/TEST SPLIT THEN SMOTE
Binary classification: INTERMEDIARY=0, HIGH=1
SMOTE applied ONLY to training data (no data leakage)

Features: 32  |  Rows: 1309
Class distribution:
CVD_Risk_Encoded
INTERMEDIARY    581
HIGH            728
Name: count, dtype: int64

Train (pre-SMOTE):  1047  {1: 582, 0: 465}
Test  (real only):  262  {1: 146, 0: 116}

Train (post-SMOTE): 1164  {0: 582, 1: 582}
Block 8 complete


In [9]:
# BLOCK 9: SAVE PROCESSED DATA
print('\n' + '='*60)
print('SAVING PROCESSED DATA')
print('='*60)

processed_dir = '../data/processed/'

df.to_csv(f'{processed_dir}processed_data_original.csv', index=False)
print(f'Saved: processed_data_original.csv ({df.shape[0]} rows, {df.shape[1]} cols)')

df.to_csv(f'{processed_dir}processed_data_clean.csv', index=False)
print(f'Saved: processed_data_clean.csv')

try:
    df_smote_save = pd.DataFrame(X_resampled, columns=feature_cols)
    df_smote_save['CVD_Risk_Encoded'] = y_resampled
    df_smote_save.to_csv(f'{processed_dir}processed_data_smote.csv', index=False)
    print(f'Saved: processed_data_smote.csv ({df_smote_save.shape[0]} train rows, SMOTE applied)')
except NameError:
    print('WARNING: X_resampled not found.')

import joblib
joblib.dump(scaler, f'{processed_dir}scaler.pkl')
print('Saved: scaler.pkl')

print('\n' + '='*60)
print('FEATURE ENGINEERING COMPLETE')
print('='*60)
print('  1. Binary classification: HIGH vs INTERMEDIARY')
print('  2. LOW excluded by design (documented limitation)')
print('  3. SMOTE only on training split')
print('  4. No duplicate columns')
print('  5. CVD Risk Score retained (legitimate clinical feature)')



SAVING PROCESSED DATA
Saved: processed_data_original.csv (1309 rows, 33 cols)
Saved: processed_data_clean.csv
Saved: processed_data_smote.csv (1164 train rows, SMOTE applied)
Saved: scaler.pkl

FEATURE ENGINEERING COMPLETE
  1. Binary classification: HIGH vs INTERMEDIARY
  2. LOW excluded by design (documented limitation)
  3. SMOTE only on training split
  4. No duplicate columns
  5. CVD Risk Score retained (legitimate clinical feature)
